# Detección de marcas y productos de Sephora en transcripciones de videos de TikTok

- Se basa en los datos JSON (para nombres de marca y productos).
- Se habilita coincidencias parciales para nombres de marcas como Rare Beauty by Selena Gomez, que suele aparecer sólo como Rare Beauty 
- Para la marca de la propia tienda online su línea de productos "Sephora Collection", se utiliza un nivel más extricto de similitud, ya que se mencionará más veces Sephora como la tienda que como la marca.  

In [ ]:
import os
import re
import json
import ast
import numpy as np
import pandas as pd
import spacy
from tqdm import tqdm
from collections import defaultdict
from spacy.matcher import PhraseMatcher
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Rutas de los archivos
sentences_transcriptions_path = r"C:\Users\sandr\Documents\scrp_tiktok_tfg\analysis\nlp\products and brands detection\sentences_transcriptions.xlsx"

: 

In [ ]:
# Sacamos listado de todas las marcas que hay en Sephora, valores únicos (col brand del csv: sephora_website_cleaned en "C:\Users\sandr\Documents\scrp_tiktok_tfg\data\clean_data\sephora_website_cleaned.csv")

sephora_data = pd.read_csv("C:\\Users\\sandr\\Documents\\scrp_tiktok_tfg\\data\\clean_data\\sephora_website_cleaned.csv")
unique_brands = sephora_data['brand'].unique()

#JSON: # Sacamos un JSON con las marcas y sus productos (columna title).
json_dir = "C:\\Users\\sandr\\Documents\\scrp_tiktok_tfg\\analysis\\nlp"
if not os.path.exists(json_dir):
    os.makedirs(json_dir)

brand_products = {}
for brand in tqdm(unique_brands, desc="Procesando marcas"):
    brand_data = sephora_data[sephora_data['brand'] == brand]
    products = brand_data['title'].tolist()
    brand_products[brand] = products

with open(f"{json_dir}/all_brands_products.json", 'w', encoding='utf-8') as f:
    json.dump(brand_products, f, ensure_ascii=False, indent=4)

print(f"Se ha creado un archivo JSON con {len(unique_brands)} marcas")



Procesando marcas: 100%|██████████| 147/147 [00:00<00:00, 962.31it/s] 

Se ha creado un archivo JSON con 147 marcas


: 

In [ ]:
# MARCAS CON SUS PRODUCTOS, JSON Y JSON CON PATRONES REGEX
brands_products_path = r"C:\Users\sandr\Documents\scrp_tiktok_tfg\analysis\nlp\products and brands detection\all_brands_products.json"

# Cargar los JSONs
with open(brands_products_path, 'r', encoding='utf-8') as f:
    brands_products = json.load(f)


: 

In [ ]:
# VIDEOS Y TRANSCRIPCIONES POR FRASES. 
input_file = r"C:\Users\sandr\Documents\scrp_tiktok_tfg\etl_and_eda\data\clean_data\url_data_cleaned.xlsx"
output_dir = r"C:\Users\sandr\Documents\scrp_tiktok_tfg\analysis\nlp\products and brands detection"
output_file = "sentences_transcriptions.xlsx"
output_path = os.path.join(output_dir, output_file)

print(f"Reading file: {input_file}")
df = pd.read_excel(input_file)

# Columns to use:
columns = df.columns.tolist()
url_col = columns[0] # video url
transcript_col = columns[1] # transcription

# List to store results
results = []

for _, row in df.iterrows():
    url = row[url_col]
    transcript = str(row[transcript_col])
    if pd.isna(transcript) or transcript.strip() == "":
        continue
    # Split transcript into sentences
    # This regex splits on periods, question marks, and exclamation marks
    # followed by a space or end of string
    sentences = re.split(r'(?<=[.!?])\s+|(?<=[.!?])$', transcript)
    # Remove empty sentences
    sentences = [sentence.strip() for sentence in sentences if sentence.strip()]
    # Add each sentence to results with its corresponding URL
    for sentence in sentences:
        results.append({
            'video_url': url,
            'sentence': sentence
        })

result_df = pd.DataFrame(results)
print(f"Saving {len(results)} sentences to {output_path}")
result_df.to_excel(output_path, index=False)
print("Processing complete.")

Reading file: C:\Users\sandr\Documents\scrp_tiktok_tfg\etl_and_eda\data\clean_data\url_data_cleaned.xlsx
URL column: id_urlvideo
Transcript column: transcription
Saving 6530 sentences to C:\Users\sandr\Documents\scrp_tiktok_tfg\analysis\nlp\products and brands detection\sentences_transcriptions.xlsx
Processing complete.


: 

Contamos con los siguientes datos para mejorar la precisión en la detección de marcas y productos de Sephora:

- **JSON con datos de la web de Sephora**: Contiene los nombres de las marcas y sus productos.  
- **JSON con patrones RegEx para el modelo BERT**: Define patrones específicos para identificar marcas y productos mediante coincidencia parcial.  
- **Archivo independiente con transcripciones**: Incluye las transcripciones desglosadas por frases y las URL correspondientes para rastrear el origen de cada video.  

In [ ]:
# SspaCy model for text processing
try:
    nlp = spacy.load('en_core_web_md')
except:
    # If model not installed, download it
    import subprocess
    subprocess.call(['python', '-m', 'spacy', 'download', 'en_core_web_md'])
    nlp = spacy.load('en_core_web_md')

# All paths to use: 
base_dir = r"C:\Users\sandr\Documents\scrp_tiktok_tfg\analysis\nlp\products and brands detection"
sentences_path = r"C:\Users\sandr\Documents\scrp_tiktok_tfg\analysis\nlp\products and brands detection\sentences_transcriptions.xlsx"
brands_products_path = os.path.join(os.path.dirname(base_dir), "all_brands_products.json")
output_dir = base_dir

: 

In [29]:
# Set paths appropriately
base_dir = r"C:\Users\sandr\Documents\scrp_tiktok_tfg\analysis\nlp\products and brands detection"
sentences_path = os.path.join(base_dir, "sentences_transcriptions.xlsx")
brands_products_path = os.path.join(os.path.dirname(base_dir), "all_brands_products.json")
output_dir = base_dir

print(f"Reading sentences from: {sentences_path}")
print(f"Reading brands/products from: {brands_products_path}")
print(f"Output directory: {output_dir}")

# Load data
print("Loading data...")
try:
    sentences_df = pd.read_excel(sentences_path)
    print(f"Loaded {len(sentences_df)} sentences")
    
    with open(brands_products_path, 'r', encoding='utf-8') as f:
        brands_products = json.load(f)
    print(f"Loaded data for {len(brands_products)} brands")
    
    # Clean up brands data - ensure all keys are strings
    cleaned_brands_products = {}
    for brand, products in brands_products.items():
        if not isinstance(brand, str):
            brand = str(brand)
            
        # Explicitly handle common brand name issues
        if brand.lower() == "rare beauty":
            brand = "Rare Beauty by Selena Gomez"  # Ensure consistent naming
        
        # Ensure products are all strings too
        cleaned_products = []
        for product in products:
            if product is not None:  # Skip None values
                if not isinstance(product, str):
                    product = str(product)
                cleaned_products.append(product)
        
        cleaned_brands_products[brand] = cleaned_products
    
    brands_products = cleaned_brands_products  # Replace with cleaned version
    
except Exception as e:
    print(f"Error loading data: {e}")
    raise

# Function to normalize text for comparison
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    # Convert to lowercase and remove special characters
    normalized = re.sub(r'[^\w\s]', ' ', text.lower())
    # Remove extra spaces
    normalized = re.sub(r'\s+', ' ', normalized).strip()
    return normalized

# Create a product to brand mapping
print("Creating product to brand mapping...")
product_to_brand = {}
for brand, products in brands_products.items():
    for product in products:
        normalized_product = normalize_text(product)
        product_to_brand[normalized_product] = brand

# Function to handle alternate brand names or mistyped brands
def get_brand_variants(brand):
    brand_lower = brand.lower()
    variants = [brand_lower]
    
    # Add possessive forms
    variants.append(f"{brand_lower}'s")
    variants.append(f"{brand_lower}s")
    
    # Handle common brand variations
    brand_specific_variants = {
        "benefit cosmetics": ["benefit"],
        "nars": ["nars cosmetics"],
        # Note: Removed "it" as a variant for "it cosmetics" to prevent false positives
        "shiseido": ["shiseido's"],
        "real techniques": ["real technique"],
        "charlotte tilbury": ["charlotte's", "tilbury"]
    }
    
    if brand_lower in brand_specific_variants:
        variants.extend(brand_specific_variants[brand_lower])
    
    return variants

# Function to check if a brand is mentioned in a sentence with partial matching
def is_brand_mentioned(sentence, brand):
    # Check if inputs are valid strings
    if not isinstance(sentence, str) or not isinstance(brand, str):
        return False
    
    # Special handling for common words that are also brand names
    if brand.lower() == "it cosmetics":
        # Make sure we're not matching "it" as a pronoun
        # Only match "it cosmetics" exactly
        return re.search(r'\bit\s+cosmetics\b', sentence.lower()) is not None
    
    # Get all possible variants of the brand name
    brand_variants = get_brand_variants(brand)
    
    # Special cases requiring strict matching
    strict_match_brands = [
        "sephora collection",
        "then i met you",
        "the ordinary",
        "it cosmetics",  # Added here to ensure strict matching
        "too faced",
        "milk makeup",
        "glow recipe",
        "lys beauty",   
        "cay skin"  
    ]
    
    # Special cases requiring custom handling
    if brand.lower() in strict_match_brands:
        # Use exact phrase matching with word boundaries
        for variant in brand_variants:
            if re.search(r'\b' + re.escape(variant) + r'\b', sentence.lower()):
                return True
        return False
        
    # Special case for Rare Beauty (allow partial matching)
    elif brand.lower() == "rare beauty by selena gomez":
        return re.search(r'\brare\s+beauty\b', sentence.lower()) is not None
    elif brand.lower() == "charlotte tilbury":
        return re.search(r'\bcharlotte\s+tilbury\b', sentence.lower()) is not None
    elif brand.lower() == "huda beauty":
        return re.search(r'\bhuda\s+beauty\b', sentence.lower()) is not None
    elif brand.lower() == "kvd beauty":
        return re.search(r'\bkvd\b', sentence.lower()) is not None
    
    # For other brands, allow partial matches but with word boundaries
    for variant in brand_variants:
        brand_words = variant.split()
        # For brands with multiple words, check if the main parts are present
        if len(brand_words) > 1:
            # Extract significant words (not common words like "the", "and", etc.)
            main_words = [word for word in brand_words if len(word) > 3 and word not in ["the", "and", "with", "by", "of", "for"]]
            if main_words:
                # Build a pattern that allows words between the main words
                pattern = r'\b' + r'.*\b'.join(main_words) + r'\b'
                if re.search(pattern, sentence.lower()):
                    return True
        else:
            # For single word brands
            if re.search(r'\b' + re.escape(variant) + r'\b', sentence.lower()):
                return True
    
    return False

# Extract brand names directly from text using possessive forms and known patterns
def extract_direct_brand_mentions(sentence):
    direct_mentions = []
    
    # List of common English pronouns and conjunctions to avoid false matches
    common_words = ["it", "the", "and", "but", "or", "so", "if", "when", "that", "this","I","love"]
    
    # Look for possessive forms (Brand's Product)
    possessive_pattern = r'(\b[A-Z][a-zA-Z]+\'s\b|\b[A-Z][a-zA-Z]+s\b)'
    possessive_matches = re.findall(possessive_pattern, sentence)
    
    for match in possessive_matches:
        # Remove the 's or s ending
        if match.endswith("'s"):
            brand_name = match[:-2]
        elif match.endswith("s"):
            brand_name = match[:-1]
        else:
            brand_name = match
        
        # Skip if it's a common word
        if brand_name.lower() in common_words:
            continue
            
        # Check if this is a known brand or close to a known brand
        for known_brand in brands_products.keys():
            if brand_name.lower() in known_brand.lower() or known_brand.lower() in brand_name.lower():
                direct_mentions.append(known_brand)
                break
    
    # Look for "of Brand" pattern (common in Spanish translations)
    of_pattern = r'of\s+([A-Z][a-zA-Z\s]+)'
    of_matches = re.findall(of_pattern, sentence)
    
    for match in of_matches:
        # Skip if it's a common word
        if match.lower() in common_words:
            continue
            
        for known_brand in brands_products.keys():
            # Make sure we have a substantial match, not just a single letter or common word
            if (match.lower() in known_brand.lower() and len(match) > 3) or known_brand.lower() in match.lower():
                direct_mentions.append(known_brand)
                break
    
    # Look for "Brand + product type" pattern
    # e.g., "Benefit mascara", "Nars foundation"
    product_types = ["foundation", "lipstick", "mascara", "powder", "palette", "brush", "serum", "moisturizer", "cream"]
    
    for product_type in product_types:
        pattern = rf'\b([A-Z][a-zA-Z]+)\s+{product_type}\b'
        matches = re.findall(pattern, sentence)
        
        for match in matches:
            # Skip if it's a common word
            if match.lower() in common_words:
                continue
                
            for known_brand in brands_products.keys():
                if match.lower() in known_brand.lower() or known_brand.lower() in match.lower():
                    direct_mentions.append(known_brand)
                    break
    
    return direct_mentions

# Function to check if a product is mentioned in a sentence
def is_product_mentioned(sentence, product, brand):
    # Check if inputs are valid
    if not isinstance(product, str):
        return False
    if not isinstance(sentence, str):
        return False
        
    normalized_sentence = normalize_text(sentence)
    normalized_product = normalize_text(product)
    
    # Products containing "love" need stricter matching to avoid false positives
    if "love" in normalized_product.lower():
        # Require exact matching for products with "love"
        # Only match if at least 80% of the product words are present in sequence
        product_words = normalized_product.split()
        if len(product_words) > 2:
            # Look for sequences of words that match most of the product name
            matches = 0
            sequence_length = int(len(product_words) * 0.8)
            for i in range(len(product_words) - sequence_length + 1):
                sequence = ' '.join(product_words[i:i+sequence_length])
                if sequence in normalized_sentence:
                    matches += 1
            return matches > 0
    
    # Skip very short product names to reduce false positives
    if len(normalized_product.split()) < 2 and len(normalized_product) < 5:
        # Exception: Allow short product names if they're distinctive enough 
        # and the brand is mentioned in the sentence
        if normalized_product in normalized_sentence:
            # Check if brand or its variants are mentioned in the sentence
            brand_variants = get_brand_variants(brand)
            for variant in brand_variants:
                if variant in normalized_sentence:
                    return True
        return False
    
    # Products that need special handling to avoid false positives
    problematic_products = [
        "brush", "foundation", "mascara", "powder", "lipstick", "palette", 
        "serum", "moisturizer", "cream", "cleanser", "liner", "balm", "blush",
        "primer", "perfume", "fragrance", "brow", "palette", "eyeshadow"
    ]
    
    # For generic-sounding products, require the brand to be mentioned too
    for generic_term in problematic_products:
        if generic_term in normalized_product and len(normalized_product.split()) < 3:
            # Only match if the brand is also mentioned in the sentence
            brand_variants = get_brand_variants(brand)
            brand_mentioned = False
            for variant in brand_variants:
                if variant in normalized_sentence:
                    brand_mentioned = True
                    break
            
            if not brand_mentioned:
                return False
    
    # For longer product names, create a more flexible pattern
    words = normalized_product.split()
    significant_words = [word for word in words if len(word) > 3 and word not in ["the", "and", "with", "for", "from", "size", "mini", "set"]]
    
    # If we have significant words, check if they appear in the sentence
    if significant_words:
        # Check if at least 60% of significant words appear in the sentence
        sentence_words = normalized_sentence.split()
        matching_words = sum(1 for word in significant_words if word in sentence_words)
        
        # More distinctive products need fewer matches
        threshold = max(2, len(significant_words) * 0.6)
        
        # If the product contains the brand name, reduce the threshold slightly
        # because brand might be mentioned separately
        if brand.lower() in normalized_product:
            threshold = max(1, threshold - 1)
        
        # Special case for products with "brow" or other generic terms
        if any(generic in normalized_product for generic in problematic_products):
            # Higher threshold for generic products to reduce false positives
            threshold = max(threshold, len(significant_words) * 0.7)
            
        return matching_words >= threshold
    
    # Fallback for products with only short words - require exact match
    return normalized_product in normalized_sentence

# Process sentences and detect brands and products
print("Detecting brands and products in sentences...")
results = []

for _, row in tqdm(sentences_df.iterrows(), total=len(sentences_df)):
    try:
        video_url = row['video_url']
        sentence = row['sentence']
        
        # Ensure sentence is a string
        if not isinstance(sentence, str):
            if pd.isna(sentence):
                continue  # Skip NaN values
            sentence = str(sentence)  # Convert to string if it's another type (like float)
        
        found_brands = []
        found_products = []
        
        # First try to extract direct brand mentions from patterns
        direct_mentions = extract_direct_brand_mentions(sentence)
        for brand in direct_mentions:
            if brand not in found_brands:
                found_brands.append(brand)
        
        # Then check for brand mentions
        for brand in brands_products.keys():
            # Skip the sentence if it contains custom stop words that might cause false positives
            should_skip = False
            
            # Special handling for "it cosmetics" and other brands with common words
            if brand.lower() == "it cosmetics":
                # Only detect "it cosmetics" if both words appear together
                if not re.search(r'\bit\s+cosmetics\b', sentence.lower()):
                    should_skip = True
            
            # Special handling for LYS and CAY
            if brand.lower() in ["lys beauty", "cay skin"]:
                brand_initial = brand.lower().split()[0]  # Get first word (LYS or CAY)
                # Only consider it a match if "beauty" or "skin" also appears nearby
                second_word = brand.lower().split()[1]  # "beauty" or "skin"
                if not (re.search(r'\b' + re.escape(brand_initial) + r'\b', sentence.lower()) and 
                        re.search(r'\b' + re.escape(second_word) + r'\b', sentence.lower())):
                    should_skip = True
            
            if not should_skip and is_brand_mentioned(sentence, brand) and brand not in found_brands:
                found_brands.append(brand)
        
        # Check for product mentions - for each brand we found
        for brand in found_brands:
            for product in brands_products[brand]:
                if is_product_mentioned(sentence, product, brand):
                    # Check if this product might be a false positive
                    # (e.g. generic product name with no brand context)
                    is_likely_false_positive = False
                    
                    # For very generic products like "Brow Serum", be extra careful
                    normalized_product = normalize_text(product)
                    generic_terms = ["brow", "serum", "brush", "mascara"]
                    
                    if len(normalized_product.split()) <= 2 and any(
                        generic in normalized_product for generic in generic_terms
                    ):
                        # Check if the brand name or product-specific terms are mentioned
                        unique_product_terms = set(normalized_product.split()) - set(
                            generic_terms + ["the", "and", "with"]
                        )
                        
                        if not unique_product_terms:
                            # If no unique terms in product name, require brand to be clearly mentioned
                            brand_variants = get_brand_variants(brand)
                            brand_found = False
                            for variant in brand_variants:
                                if re.search(r'\b' + re.escape(variant) + r'\b', sentence.lower()):
                                    brand_found = True
                                    break
                            
                            if not brand_found:
                                is_likely_false_positive = True
                    
                    if not is_likely_false_positive:
                        # Check for specific product matches
                        if "hubba brow" in normalized_product.lower() and "huba brow" in sentence.lower():
                            found_products.append({
                                'product': product,
                                'brand': brand
                            })
                        else:
                            found_products.append({
                                'product': product,
                                'brand': brand
                            })
        
        # Also check all products to find those whose brand wasn't explicitly mentioned
        if not found_products:  # Only if we haven't found any products yet
            for brand, products in brands_products.items():
                if brand not in found_brands:  # Skip brands we already processed
                    for product in products:
                        # Don't check generic products without brand context
                        normalized_product = normalize_text(product)
                        generic_terms = ["brow", "serum", "brush", "mascara", "foundation", "lipstick"]
                        
                        # Skip generic products unless specifically mentioned with their brand
                        if any(generic in normalized_product for generic in generic_terms) and len(normalized_product.split()) <= 2:
                            continue
                        
                        if is_product_mentioned(sentence, product, brand):
                            found_products.append({
                                'product': product,
                                'brand': brand
                            })
        
        # Add to results if we found any brands or products
        if found_brands or found_products:
            results.append({
                'video_url': video_url,
                'sentence': sentence,
                'mentioned_brands': found_brands,
                'mentioned_products': [p['product'] for p in found_products],
                'product_brands': [p['brand'] for p in found_products]
            })
    except Exception as e:
        print(f"Error processing row: {e}")
        continue

# Create DataFrame with results
print(f"Found mentions in {len(results)} sentences")
results_df = pd.DataFrame(results)

# Save results
output_path = os.path.join(output_dir, "brand_product_mentions.xlsx")
results_df.to_excel(output_path, index=False)
print(f"Results saved to {output_path}")



print("\nAnalysis complete!")

Reading sentences from: C:\Users\sandr\Documents\scrp_tiktok_tfg\analysis\nlp\products and brands detection\sentences_transcriptions.xlsx
Reading brands/products from: C:\Users\sandr\Documents\scrp_tiktok_tfg\analysis\nlp\all_brands_products.json
Output directory: C:\Users\sandr\Documents\scrp_tiktok_tfg\analysis\nlp\products and brands detection
Loading data...
Loaded 6530 sentences
Loaded data for 147 brands
Creating product to brand mapping...
Detecting brands and products in sentences...


100%|██████████| 6530/6530 [05:35<00:00, 19.44it/s]


Found mentions in 1396 sentences
Results saved to C:\Users\sandr\Documents\scrp_tiktok_tfg\analysis\nlp\products and brands detection\brand_product_mentions.xlsx

Analysis complete!


In [30]:
# Save everything to a single Excel file with multiple sheets
output_path = os.path.join(output_dir, "brand_product_mentions.xlsx")
    
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    results_df.to_excel(writer, sheet_name='Detecciones', index=False)
    brand_counts_df.to_excel(writer, sheet_name='Resumen Marcas', index=False)
    product_counts_df.to_excel(writer, sheet_name='Resumen Productos', index=False)
    
print(f"Results and summaries saved to {output_path}")

Results and summaries saved to C:\Users\sandr\Documents\scrp_tiktok_tfg\analysis\nlp\products and brands detection\brand_product_mentions.xlsx


In [ ]:
# Ruta del archivo Excel original
input_file = r"C:\Users\sandr\Documents\scrp_tiktok_tfg\analysis\nlp\products and brands detection\brand_product_mentions.xlsx"
output_file = r"C:\Users\sandr\Documents\scrp_tiktok_tfg\analysis\nlp\products and brands detection\brand_product_mentions_cleaned.xlsx"

print(f"Limpiando filas sin detecciones del archivo: {input_file}")

# Cargar el archivo Excel original
try:
    # Cargar todas las hojas del Excel
    detecciones_df = pd.read_excel(input_file, sheet_name='Detecciones')
    
    # Intentar cargar otras hojas si existen
    try:
        resumen_marcas_df = pd.read_excel(input_file, sheet_name='Resumen Marcas')
        resumen_productos_df = pd.read_excel(input_file, sheet_name='Resumen Productos')
        has_summary_sheets = True
    except:
        has_summary_sheets = False
    
    print(f"Cargados {len(detecciones_df)} registros de detecciones")
    
    # Convertir cadenas de texto que representan listas a listas reales si es necesario
    if isinstance(detecciones_df['mentioned_brands'].iloc[0], str):
        detecciones_df['mentioned_brands'] = detecciones_df['mentioned_brands'].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )
    
    if isinstance(detecciones_df['mentioned_products'].iloc[0], str):
        detecciones_df['mentioned_products'] = detecciones_df['mentioned_products'].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )
        
    if 'product_brands' in detecciones_df.columns and isinstance(detecciones_df['product_brands'].iloc[0], str):
        detecciones_df['product_brands'] = detecciones_df['product_brands'].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )
    
except Exception as e:
    print(f"Error al cargar el archivo: {e}")
    raise

# Antes de guardar, eliminar filas sin detecciones
print("Eliminando filas sin detecciones...")
# Contar filas originales
original_row_count = len(detecciones_df)

# Filtrar filas donde no hay marcas ni productos detectados
cleaned_df = detecciones_df[
    (detecciones_df['mentioned_brands'].apply(lambda x: len(x) > 0)) | 
    (detecciones_df['mentioned_products'].apply(lambda x: len(x) > 0))
]

# Contar filas eliminadas
removed_rows = original_row_count - len(cleaned_df)
print(f"Se eliminaron {removed_rows} filas sin detecciones.")
print(f"Quedan {len(cleaned_df)} filas con detecciones.")

# Generar nuevos resúmenes después de la limpieza
# Recuento de menciones de marcas
brand_counts = defaultdict(int)
for brands in cleaned_df['mentioned_brands']:
    for brand in brands:
        brand_counts[brand] += 1

# Recuento de menciones de productos
product_counts = defaultdict(int)
for i, row in cleaned_df.iterrows():
    for j, product in enumerate(row['mentioned_products']):
        if j < len(row['product_brands']):
            product_brand = f"{product} ({row['product_brands'][j]})"
            if product_brand in product_counts:
                product_counts[product_brand] += 1
            else:
                product_counts[product_brand] = 1

# Convertir a DataFrames
brand_counts_df = pd.DataFrame({
    'Brand': list(brand_counts.keys()),
    'Mentions': list(brand_counts.values())
}).sort_values('Mentions', ascending=False)

product_counts_df = pd.DataFrame({
    'Product (Brand)': list(product_counts.keys()),
    'Mentions': list(product_counts.values())
}).sort_values('Mentions', ascending=False)

print(f"\nDespués de eliminar filas vacías:")
print(f"Top 10 marcas más mencionadas:")
print(brand_counts_df.head(10).to_string())
print(f"\nTop 10 productos más mencionados:")
print(product_counts_df.head(10).to_string())

# Guardar todo en un único archivo Excel con múltiples hojas
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    cleaned_df.to_excel(writer, sheet_name='Detecciones', index=False)
    brand_counts_df.to_excel(writer, sheet_name='Resumen Marcas', index=False)
    product_counts_df.to_excel(writer, sheet_name='Resumen Productos', index=False)
    
    # Guardar también los datos originales para comparación
    detecciones_df.to_excel(writer, sheet_name='Detecciones Originales', index=False)

print(f"\nResultados limpios guardados en: {output_file}")
print("\nProceso de limpieza completado.")

Limpiando filas sin detecciones del archivo: C:\Users\sandr\Documents\scrp_tiktok_tfg\analysis\nlp\products and brands detection\brand_product_mentions.xlsx
Cargados 1396 registros de detecciones
Eliminando filas sin detecciones...
Se eliminaron 0 filas sin detecciones.
Quedan 1396 filas con detecciones.

Después de eliminar filas vacías:
Top 10 marcas más mencionadas:
                          Brand  Mentions
12            Charlotte Tilbury       179
16  Rare Beauty by Selena Gomez       114
10                         DIOR        79
21                  HUDA BEAUTY        71
20                    Hourglass        68
3             Benefit Cosmetics        67
36                   PATRICK TA        54
29              MAKEUP BY MARIO        49
5                          NARS        49
32                    Too Faced        48

Top 10 productos más mencionados:
                                         Product (Brand)  Mentions
10                           Dior Addict Lip Glow (DIOR)       1

Entonces, el archivo **brand_product_mentions_cleaned** contendrá finalmente las detecciones de marcas y productos mencionados en los videos de TikTok.  


In [ ]:
import pandas as pd
import numpy as np
import os
import re
import ast
import json
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
import spacy
from textblob import TextBlob
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/624.3 kB ? eta -:--:--
   --------------------------------------- 624.3/624.3 kB 11.7 MB/s eta 0:00:00


In [36]:
try:
    nltk.download('vader_lexicon', quiet=True)
    nltk.download('punkt', quiet=True)
    print("Recursos NLTK descargados correctamente")
except:
    print("Error descargando recursos NLTK, verificar conexión a internet")

# Cargar el archivo con las detecciones refinadas
print(f"Cargando datos desde: {input_file}")
try:
    # Cargar diferentes hojas del Excel
    detections_df = pd.read_excel(input_file, sheet_name='Detecciones')
    brands_summary_df = pd.read_excel(input_file, sheet_name='Resumen Marcas')
    products_summary_df = pd.read_excel(input_file, sheet_name='Resumen Productos')
    
    # Convertir cadenas de texto que representan listas a listas reales
    if isinstance(detections_df['mentioned_brands'].iloc[0], str):
        detections_df['mentioned_brands'] = detections_df['mentioned_brands'].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )
    
    if isinstance(detections_df['mentioned_products'].iloc[0], str):
        detections_df['mentioned_products'] = detections_df['mentioned_products'].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )
    
    if 'product_brands' in detections_df.columns and isinstance(detections_df['product_brands'].iloc[0], str):
        detections_df['product_brands'] = detections_df['product_brands'].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )
    
    print(f"Cargados {len(detections_df)} registros de detecciones")
except Exception as e:
    print(f"Error al cargar el archivo: {e}")
    raise

# 1. Evaluación de los modelos de detección
print("\n===== Evaluación del modelo de detección =====")

# Obtener estadísticas de detección
num_sentences_total = len(detections_df)
num_sentences_with_brands = sum(detections_df['mentioned_brands'].apply(len) > 0)
num_sentences_with_products = sum(detections_df['mentioned_products'].apply(len) > 0)
num_sentences_with_both = sum((detections_df['mentioned_brands'].apply(len) > 0) & 
                             (detections_df['mentioned_products'].apply(len) > 0))

# Porcentaje de frases con detecciones
percent_with_brands = (num_sentences_with_brands / num_sentences_total) * 100
percent_with_products = (num_sentences_with_products / num_sentences_total) * 100
percent_with_both = (num_sentences_with_both / num_sentences_total) * 100

print(f"Total de frases analizadas: {num_sentences_total}")
print(f"Frases con marcas detectadas: {num_sentences_with_brands} ({percent_with_brands:.2f}%)")
print(f"Frases con productos detectados: {num_sentences_with_products} ({percent_with_products:.2f}%)")
print(f"Frases con marcas y productos detectados: {num_sentences_with_both} ({percent_with_both:.2f}%)")

# Calcular ratio de detección de productos por marca
brand_product_ratio = {}
for _, row in detections_df.iterrows():
    for brand in row['mentioned_brands']:
        if brand not in brand_product_ratio:
            brand_product_ratio[brand] = {'brand_mentions': 0, 'product_mentions': 0}
        brand_product_ratio[brand]['brand_mentions'] += 1
        
        # Contar productos de esta marca que fueron mencionados
        brand_products = [i for i, b in enumerate(row['product_brands']) if b == brand]
        brand_product_ratio[brand]['product_mentions'] += len(brand_products)

# Crear DataFrame con ratio de producto/marca
brand_product_ratio_df = pd.DataFrame({
    'Brand': list(brand_product_ratio.keys()),
    'Brand_Mentions': [d['brand_mentions'] for d in brand_product_ratio.values()],
    'Product_Mentions': [d['product_mentions'] for d in brand_product_ratio.values()]
})

brand_product_ratio_df['Product_Brand_Ratio'] = brand_product_ratio_df['Product_Mentions'] / brand_product_ratio_df['Brand_Mentions']
brand_product_ratio_df = brand_product_ratio_df.sort_values('Product_Brand_Ratio', ascending=False)

print("\nTop 10 marcas con mayor ratio producto/marca:")
print(brand_product_ratio_df.head(10)[['Brand', 'Brand_Mentions', 'Product_Mentions', 'Product_Brand_Ratio']].to_string(index=False))

# Guardar esta información para incluirla en el Excel final
model_evaluation_df = pd.DataFrame({
    'Metric': ['Total Sentences', 'Sentences with Brands', 'Sentences with Products', 
               'Sentences with Both', 'Brand Detection Rate', 'Product Detection Rate', 
               'Both Detection Rate'],
    'Value': [num_sentences_total, num_sentences_with_brands, num_sentences_with_products,
              num_sentences_with_both, f"{percent_with_brands:.2f}%", 
              f"{percent_with_products:.2f}%", f"{percent_with_both:.2f}%"]
})

# 2. Análisis de sentimiento - utilizamos VADER de NLTK
print("\n===== Realizando análisis de sentimiento =====")

# Inicializar el analizador de sentimiento
sia = SentimentIntensityAnalyzer()

# Función para obtener puntuación de sentimiento con VADER
def get_sentiment_vader(text):
    if not isinstance(text, str):
        return {'neg': 0, 'neu': 0, 'pos': 0, 'compound': 0}
    return sia.polarity_scores(text)

# Función para obtener puntuación de sentimiento con TextBlob (como método alternativo)
def get_sentiment_textblob(text):
    if not isinstance(text, str):
        return {'polarity': 0, 'subjectivity': 0}
    analysis = TextBlob(text)
    return {'polarity': analysis.sentiment.polarity, 'subjectivity': analysis.sentiment.subjectivity}

# Añadir puntuaciones de sentimiento a cada frase
detections_df['sentiment_vader'] = detections_df['sentence'].apply(get_sentiment_vader)
detections_df['sentiment_textblob'] = detections_df['sentence'].apply(get_sentiment_textblob)

# Extraer componentes del sentimiento en columnas separadas para facilitar el análisis
detections_df['sentiment_negative'] = detections_df['sentiment_vader'].apply(lambda x: x['neg'])
detections_df['sentiment_neutral'] = detections_df['sentiment_vader'].apply(lambda x: x['neu'])
detections_df['sentiment_positive'] = detections_df['sentiment_vader'].apply(lambda x: x['pos'])
detections_df['sentiment_compound'] = detections_df['sentiment_vader'].apply(lambda x: x['compound'])
detections_df['sentiment_polarity'] = detections_df['sentiment_textblob'].apply(lambda x: x['polarity'])
detections_df['sentiment_subjectivity'] = detections_df['sentiment_textblob'].apply(lambda x: x['subjectivity'])

# Clasificar el sentimiento basado en la puntuación compuesta
detections_df['sentiment_category'] = detections_df['sentiment_compound'].apply(
    lambda x: 'Positive' if x >= 0.05 else ('Negative' if x <= -0.05 else 'Neutral')
)

# Calcular estadísticas de sentimiento
sentiment_counts = detections_df['sentiment_category'].value_counts()
print("\nDistribución de sentimiento en frases:")
for category, count in sentiment_counts.items():
    percentage = (count / len(detections_df)) * 100
    print(f"{category}: {count} frases ({percentage:.2f}%)")

# 3. Análisis de sentimiento por marca y producto
print("\n===== Análisis de sentimiento por marca y producto =====")

# Calcular sentimiento promedio por marca
brand_sentiment = defaultdict(list)
for _, row in detections_df.iterrows():
    for brand in row['mentioned_brands']:
        brand_sentiment[brand].append(row['sentiment_compound'])

brand_sentiment_df = pd.DataFrame({
    'Brand': list(brand_sentiment.keys()),
    'Avg_Sentiment': [np.mean(scores) for scores in brand_sentiment.values()],
    'Mentions': [len(scores) for scores in brand_sentiment.values()]
})
brand_sentiment_df = brand_sentiment_df.sort_values('Avg_Sentiment', ascending=False)

print("\nTop 10 marcas con sentimiento más positivo:")
print(brand_sentiment_df.head(10)[['Brand', 'Mentions', 'Avg_Sentiment']].to_string(index=False))

print("\nMarcas con sentimiento más negativo:")
print(brand_sentiment_df.tail(10)[['Brand', 'Mentions', 'Avg_Sentiment']].to_string(index=False))

# Calcular sentimiento promedio por producto
product_sentiment = defaultdict(list)
for _, row in detections_df.iterrows():
    for i, product in enumerate(row['mentioned_products']):
        if i < len(row['product_brands']):
            brand = row['product_brands'][i]
            product_key = f"{product} ({brand})"
            product_sentiment[product_key].append(row['sentiment_compound'])

product_sentiment_df = pd.DataFrame({
    'Product': list(product_sentiment.keys()),
    'Avg_Sentiment': [np.mean(scores) for scores in product_sentiment.values()],
    'Mentions': [len(scores) for scores in product_sentiment.values()]
})
product_sentiment_df = product_sentiment_df.sort_values('Avg_Sentiment', ascending=False)

print("\nTop 10 productos con sentimiento más positivo:")
print(product_sentiment_df.head(10)[['Product', 'Mentions', 'Avg_Sentiment']].to_string(index=False))

print("\nProductos con sentimiento más negativo:")
negative_products = product_sentiment_df[product_sentiment_df['Avg_Sentiment'] < 0]
if len(negative_products) > 0:
    print(negative_products[['Product', 'Mentions', 'Avg_Sentiment']].to_string(index=False))
else:
    print("No se encontraron productos con sentimiento negativo.")

# 4. Añadir métricas adicionales
print("\n===== Calculando métricas adicionales =====")

# Contar palabras en cada frase
detections_df['word_count'] = detections_df['sentence'].apply(
    lambda x: len(x.split()) if isinstance(x, str) else 0
)

# Calcular la densidad de la mención (número de marcas y productos por palabra)
detections_df['mention_density'] = (
    (detections_df['mentioned_brands'].apply(len) + detections_df['mentioned_products'].apply(len)) / 
    detections_df['word_count']
)

# Eliminar filas con densidad infinita (si word_count es 0)
detections_df['mention_density'] = detections_df['mention_density'].replace([np.inf, -np.inf], np.nan)
detections_df['mention_density'] = detections_df['mention_density'].fillna(0)

# Calcular estadísticas de densidad de mención
avg_mention_density = detections_df['mention_density'].mean()
max_mention_density = detections_df['mention_density'].max()

print(f"Densidad media de menciones (marcas+productos por palabra): {avg_mention_density:.4f}")
print(f"Densidad máxima de menciones: {max_mention_density:.4f}")

# 5. Enriquecer los resúmenes de marcas y productos con la información de sentimiento
# Unir información de sentimiento con los resúmenes de marcas
brands_enriched_df = brands_summary_df.merge(
    brand_sentiment_df[['Brand', 'Avg_Sentiment']], 
    left_on='Brand', 
    right_on='Brand', 
    how='left'
)

# Unir información de sentimiento con los resúmenes de productos
products_enriched_df = products_summary_df.merge(
    product_sentiment_df[['Product', 'Avg_Sentiment']], 
    left_on='Product (Brand)', 
    right_on='Product', 
    how='left'
)
products_enriched_df.drop('Product', axis=1, inplace=True, errors='ignore')

# Unir ratio producto/marca al resumen de marcas
brands_enriched_df = brands_enriched_df.merge(
    brand_product_ratio_df[['Brand', 'Product_Brand_Ratio']], 
    left_on='Brand', 
    right_on='Brand', 
    how='left'
)

# 6. Guardar todos los resultados enriquecidos
print(f"\nGuardando resultados enriquecidos en: {output_file}")

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Guardar las hojas principales
    detections_df.to_excel(writer, sheet_name='Detecciones_Enriquecidas', index=False)
    brands_enriched_df.to_excel(writer, sheet_name='Resumen_Marcas_Enriquecido', index=False)
    products_enriched_df.to_excel(writer, sheet_name='Resumen_Productos_Enriquecido', index=False)
    
    # Guardar hojas con análisis adicionales
    brand_sentiment_df.to_excel(writer, sheet_name='Sentimiento_por_Marca', index=False)
    product_sentiment_df.to_excel(writer, sheet_name='Sentimiento_por_Producto', index=False)
    brand_product_ratio_df.to_excel(writer, sheet_name='Ratio_Producto_Marca', index=False)
    model_evaluation_df.to_excel(writer, sheet_name='Evaluación_Modelo', index=False)

print("\nProceso completo. El archivo enriquecido está listo para análisis.")

Recursos NLTK descargados correctamente
Cargando datos desde: C:\Users\sandr\Documents\scrp_tiktok_tfg\analysis\nlp\products and brands detection\brand_product_mentions.xlsx
Cargados 1396 registros de detecciones

===== Evaluación del modelo de detección =====
Total de frases analizadas: 1396
Frases con marcas detectadas: 1167 (83.60%)
Frases con productos detectados: 555 (39.76%)
Frases con marcas y productos detectados: 326 (23.35%)

Top 10 marcas con mayor ratio producto/marca:
                  Brand  Brand_Mentions  Product_Mentions  Product_Brand_Ratio
                   DIOR              79               187             2.367089
             TWEEZERMAN               1                 2             2.000000
           Sunday Riley               1                 1             1.000000
Anastasia Beverly Hills              17                 8             0.470588
          Iconic London               3                 1             0.333333
      Charlotte Tilbury             179 